# Regresión Lineal Múltiple
### De la recta al hiperplano: modelo, diagnóstico y decisión

**Wilson Sandoval R.**

---

**Cómo usar este cuaderno.** Está pensado para ejecutarse **de arriba hacia abajo**
(*Kernel → Restart & Run All*). El archivo `trees.csv` debe estar en la **misma
carpeta** que este notebook. No hay rutas absolutas ni `!pip install`.

Este material es **gemelo** del documento de R (`RLM_Clase.Rmd`): misma secuencia,
mismos datos, mismos resultados. La sección 22 contiene la tabla de equivalencias
y los valores de control para verificar que ambos coinciden.

**Librerías necesarias:** `pandas`, `numpy`, `matplotlib`, `scipy`, `statsmodels`.
`seaborn` es opcional (solo mejora algunos gráficos).

Para **inferencia estadística** usamos `statsmodels`, no `sklearn`:
`sklearn.linear_model.LinearRegression` ajusta el mismo modelo pero **no devuelve
errores estándar, ni estadísticos $t$, ni p-valores, ni intervalos de confianza**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
    HAY_SEABORN = True
except ImportError:
    HAY_SEABORN = False

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
pd.set_option("display.width", 120)
plt.rcParams["figure.figsize"] = (7, 4.5)
plt.rcParams["axes.grid"] = True

print("seaborn disponible:", HAY_SEABORN)

---
# PARTE 0. Nivelación: lo que debemos saber antes de la regresión múltiple

## El vocabulario mínimo

| Concepto | Qué es |
|---|---|
| **Variable respuesta** ($Y$) | Lo que queremos explicar o predecir. Es aleatoria. |
| **Variable predictora** ($X$) | Lo que usamos para explicar. En el modelo clásico se trata como fija y medida sin error. |
| **Correlación** ($r$) | Fuerza y sentido de la asociación **lineal**. Va de $-1$ a $1$ y no tiene unidades. |
| **Asociación vs. causalidad** | Que $X$ prediga bien a $Y$ no significa que la cause. La causalidad viene del diseño del estudio. |
| **Recta de regresión** | $\hat y = \hat\beta_0 + \hat\beta_1 x$: el valor **promedio** de $Y$ estimado para cada $x$. |
| **Valor observado** ($y_i$) | El dato real. |
| **Valor estimado** ($\hat y_i$) | Lo que el modelo predice. |
| **Residual** ($e_i = y_i - \hat y_i$) | Lo que el modelo **no** explicó en esa observación. |
| **Mínimos cuadrados** | El criterio para elegir $\hat\beta$: minimizar $\sum e_i^2$. |
| **$R^2$** | Proporción de la variabilidad de $Y$ que el modelo explica. |
| **Error estándar residual** ($\hat\sigma$) | En unidades de $Y$: cuánto se desvía típicamente una observación del modelo. |
| **Prueba $t$** | ¿Este coeficiente es distinguible de cero, dado el resto del modelo? |
| **Intervalo de confianza** | Rango de valores del **parámetro** compatibles con los datos. |
| **Predicción** | Un valor de $Y$ para una combinación nueva de predictoras. |

## Un ejemplo de regresión lineal simple: ruido e hipertensión

Datos representativos de los reportados en el *Journal of Sound and Vibration*
(Vol. 151, 1991, pp. 383–394). $x$ = nivel de presión acústica (dB),
$y$ = aumento de la presión arterial (mm Hg).

**Pregunta:** ¿a mayor exposición al ruido, mayor aumento de presión arterial?

In [ ]:
ruido = pd.DataFrame({
    "db": [60, 63, 65, 70, 70, 70, 80, 90, 80, 80, 85, 89, 90, 90, 90, 90, 94, 100, 100, 100],
    "pa": [ 1,  0,  1,  2,  5,  1,  4,  6,  2,  3,  5,  4,  6,  8,  4,  5,  7,   9,   7,   6],
})
ruido.head()

In [ ]:
fig, ax = plt.subplots()
ax.scatter(ruido.db, ruido.pa, s=45, color="steelblue")
b = np.polyfit(ruido.db, ruido.pa, 1)
xs = np.linspace(ruido.db.min(), ruido.db.max(), 50)
ax.plot(xs, np.polyval(b, xs), color="firebrick", lw=2)
ax.set(xlabel="Nivel de presión acústica (dB)",
       ylabel="Aumento de presión arterial (mm Hg)",
       title="Presión arterial vs. nivel de ruido")
plt.show()

print("Correlación:", round(ruido.db.corr(ruido.pa), 4))

Una correlación de $\approx 0.865$: asociación lineal positiva y fuerte.
**Fuerte no es lo mismo que causal**: es un estudio observacional y la edad, el
tiempo de exposición o la ocupación podrían estar detrás de ambas variables.

In [ ]:
m_ruido = smf.ols("pa ~ db", data=ruido).fit()
print(m_ruido.summary())

### Lectura de resultados

$$\widehat{pa} = -10.132 + 0.174\,\cdot\,db$$

* **Intercepto** $\hat\beta_0 = -10.132$: sería el aumento promedio de presión para
  $db = 0$. **No tiene interpretación práctica**: el rango observado es 60–100 dB y
  un aumento negativo de presión no significa nada aquí. El intercepto ancla la
  recta; no siempre es un resultado sustantivo.
* **Pendiente** $\hat\beta_1 = 0.174$: por cada decibel adicional, el aumento
  promedio de presión **crece 0.174 mm Hg** (10 dB más $\approx$ 1.7 mm Hg más).
* **Error estándar residual** $\hat\sigma = 1.318$ mm Hg.
* $R^2 = 0.7483$: el ruido explica el **74.8 %** de la variabilidad observada.


In [ ]:
print(m_ruido.conf_int().rename(columns={0: "IC 2.5%", 1: "IC 97.5%"}))

La prueba $t$ de la pendiente contrasta $H_0:\beta_1 = 0$ (el ruido no aporta
información lineal) contra $H_1:\beta_1\neq 0$. Con $p < 0.001$ rechazamos $H_0$.
El IC 95 % no contiene el cero, lo cual dice lo mismo, **pero además informa de la
magnitud plausible del efecto**, que es lo que sirve para decidir.

## El puente hacia la regresión múltiple

$$Y = \beta_0 + \beta_1 X + \epsilon
\qquad\longrightarrow\qquad
Y = \beta_0 + \beta_1X_1 + \beta_2X_2 + \cdots + \beta_kX_k + \epsilon$$

Los conceptos sobreviven intactos. Lo que cambia —y es el corazón de esta clase—
es la **interpretación de cada coeficiente**.

---
# 1. Motivación: cuando una sola variable no alcanza

> **Pregunta orientadora:** ¿podemos explicar y predecir el volumen de madera de
> un árbol usando **simultáneamente** su diámetro y su altura?

Medir el volumen de madera de un árbol en pie es caro: hay que talarlo. Medir el
diámetro es trivial y la altura es medible con un clinómetro. Si un modelo predice
bien el volumen a partir de esas dos medidas, el inventario forestal completo
cambia de costo.

## Los datos: `trees`

31 cerezos negros (*black cherry*) talados.

| Variable | Descripción | Unidad |
|---|---|---|
| `Girth` | **Diámetro** medido a 4 pies 6 pulgadas del suelo. El nombre original (*girth* = circunferencia) **está mal puesto en el conjunto de datos**: los valores son diámetros. | pulgadas |
| `Height` | Altura del árbol | pies |
| `Volume` | Volumen de madera | pies cúbicos |

In [ ]:
trees = pd.read_csv("trees.csv")

# El CSV trae una primera columna con el índice de fila exportado desde R.
# En pandas aparece como 'Unnamed: 0'. Hay que eliminarla de forma segura:
trees = trees.loc[:, ~trees.columns.str.match(r"^Unnamed")]

# Índice 1..31 para hablar el MISMO idioma que R (que indexa desde 1)
trees.index = range(1, len(trees) + 1)
trees.index.name = "obs"

trees.head()

In [ ]:
print("Dimensiones:", trees.shape)
print()
trees.info()
print()
print("Valores faltantes por columna:")
print(trees.isna().sum())

In [ ]:
trees.describe()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
for ax, v in zip(axes, trees.columns):
    ax.hist(trees[v], bins=10, color="steelblue", edgecolor="white")
    ax.set_title(v)
fig.suptitle("Distribución de las tres variables")
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
for ax, v in zip(axes, trees.columns):
    ax.boxplot(trees[v], patch_artist=True,
               boxprops=dict(facecolor="steelblue", alpha=.5),
               flierprops=dict(markerfacecolor="firebrick", marker="o"))
    ax.set_title(v); ax.set_xticks([])
fig.suptitle("Diagramas de caja")
plt.tight_layout(); plt.show()

`Volume` es claramente asimétrica a la derecha y hay un árbol muy grande
(77 pies³). **Anotemos esa observación mentalmente**: volveremos a ella en el
análisis de influencia.

In [ ]:
trees.corr().round(4)

In [ ]:
if HAY_SEABORN:
    sns.pairplot(trees, height=2.2, plot_kws=dict(color="steelblue", s=30))
    plt.suptitle("Matriz de dispersión", y=1.02)
else:
    pd.plotting.scatter_matrix(trees, figsize=(7, 7), color="steelblue", diagonal="hist")
plt.show()

Tres lecturas:

1. **`Volume`–`Girth`: $r = 0.967$.** Relación muy fuerte. El diámetro por sí solo
   ya explicaría casi todo.
2. **`Volume`–`Height`: $r = 0.598$.** Moderada, con mucha dispersión.
3. **`Girth`–`Height`: $r = 0.519$.** ⚠️ **Las dos predictoras están correlacionadas
   entre sí.** Este hecho es *el* motivo por el que un coeficiente en regresión
   múltiple no significa lo mismo que en regresión simple. Lo trabajamos en la
   sección 3.

In [ ]:
pares = [("Girth", "Volume"), ("Height", "Volume"), ("Girth", "Height")]
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, (x, y) in zip(axes, pares):
    ax.scatter(trees[x], trees[y], s=40, color="steelblue")
    b = np.polyfit(trees[x], trees[y], 1)
    xs = np.linspace(trees[x].min(), trees[x].max(), 50)
    ax.plot(xs, np.polyval(b, xs), color="firebrick", lw=2)
    ax.set(xlabel=x, ylabel=y, title=f"{y} ~ {x}")
plt.tight_layout(); plt.show()

> 🎓 **Pausa para el estudiante.** Antes de ajustar nada, escriba su respuesta:
>
> 1. ¿Qué signo espera para el coeficiente de `Girth`? ¿Y para `Height`?
> 2. Un árbol es aproximadamente un cono o un cilindro. Si $V \propto d^2 h$,
>    ¿espera que la relación entre `Volume` y `Girth` sea **exactamente** una
>    recta? Guarde su respuesta: la retomamos en la sección 18.

---
# 2. El modelo de regresión lineal múltiple

$$Y_i = \beta_0 + \beta_1 X_{1i} + \beta_2 X_{2i} + \epsilon_i,\qquad i=1,\dots,n$$

Para nuestros datos:

$$Volume_i = \beta_0 + \beta_1\,Girth_i + \beta_2\,Height_i + \epsilon_i$$

Ya no ajustamos una **recta** sino un **plano** en el espacio
$(Girth, Height, Volume)$ —y con $k$ predictoras, un hiperplano en $k+1$ dimensiones.

## El intercepto $\beta_0$

Es el valor promedio de $Y$ cuando **todas** las predictoras valen cero: un árbol
de 0 pulgadas y 0 pies. **No existe**, y está muy lejos del rango observado
($Girth\in[8.3,20.6]$, $Height\in[63,87]$). Aquí es un parámetro de ajuste, no un
resultado interpretable.

## El coeficiente parcial $\beta_j$ — la idea central de la clase

> $\beta_1$ es el cambio promedio esperado en $Y$ por un aumento de **una unidad**
> en $X_1$, **manteniendo constantes las demás variables del modelo**.

Esa última cláusula lo cambia todo. En regresión simple $\beta_1$ mezcla el efecto
directo de $X_1$ **y** el efecto de todo lo que varía junto con $X_1$. Al incluir
$X_2$, el coeficiente se "limpia": es un **efecto parcial** o **ajustado**.

| Modelo | Coef. de `Girth` | Lectura |
|---|---|---|
| `Volume ~ Girth` | $5.066$ | Árboles que difieren 1 pulgada **sin importar su altura**. Parte de esa diferencia es que los gruesos **también tienden a ser altos**. |
| `Volume ~ Girth + Height` | $4.708$ | Árboles que difieren 1 pulgada **con la misma altura**. |

La diferencia $5.066 - 4.708 = 0.358$ es justamente la parte del "efecto" del
diámetro que en realidad era altura disfrazada.

---
# 3. ¿Qué significa "controlar por otras variables"?

"Manteniendo las demás constantes" no es una fórmula retórica: es una operación
concreta que podemos **ver**.

In [ ]:
m_simple   = smf.ols("Volume ~ Girth", data=trees).fit()
m_multiple = smf.ols("Volume ~ Girth + Height", data=trees).fit()

pd.DataFrame({
    "modelo":      ["Volume ~ Girth", "Volume ~ Girth + Height"],
    "coef_Girth":  [m_simple.params["Girth"], m_multiple.params["Girth"]],
    "ee_Girth":    [m_simple.bse["Girth"],    m_multiple.bse["Girth"]],
    "R2_ajustado": [m_simple.rsquared_adj,    m_multiple.rsquared_adj],
})

## Por qué cambia el coeficiente: regresión parcial "a mano"

El coeficiente de `Girth` en el modelo múltiple se obtiene en tres pasos. Esto
**es** lo que significa "controlar por `Height`".

In [ ]:
# Paso 1: quitarle a Girth todo lo que Height puede explicar de él
girth_residual  = smf.ols("Girth  ~ Height", data=trees).fit().resid

# Paso 2: quitarle a Volume todo lo que Height puede explicar de él
volume_residual = smf.ols("Volume ~ Height", data=trees).fit().resid

# Paso 3: regresar un residuo contra el otro
parcial = sm.OLS(volume_residual, sm.add_constant(girth_residual)).fit()

print("Pendiente de la regresión parcial :", round(parcial.params.iloc[1], 6))
print("Coeficiente de Girth en el modelo :", round(m_multiple.params['Girth'], 6))

Son **idénticos** (teorema de Frisch–Waugh–Lovell). El coeficiente parcial de
`Girth` es la relación entre *la parte de `Volume` que `Height` no explica* y *la
parte de `Girth` que `Height` no explica*: **la relación que queda después de
descontar la altura de ambos lados**.

In [ ]:
fig, ax = plt.subplots()
ax.scatter(girth_residual, volume_residual, s=45, color="steelblue")
b = np.polyfit(girth_residual, volume_residual, 1)
xs = np.linspace(girth_residual.min(), girth_residual.max(), 50)
ax.plot(xs, np.polyval(b, xs), color="firebrick", lw=2)
ax.axhline(0, ls="--", c="grey"); ax.axvline(0, ls="--", c="grey")
ax.set(xlabel="Girth ajustado por Height (residuos)",
       ylabel="Volume ajustado por Height (residuos)",
       title="Regresión parcial: la pendiente ES el coeficiente parcial de Girth")
plt.show()

## Tres advertencias

1. **El coeficiente parcial depende del modelo.** Si agrego una tercera predictora,
   el de `Girth` puede volver a cambiar. No es "el efecto del diámetro": es "el
   efecto del diámetro **en este modelo**".
2. **"Manteniendo constante" puede ser físicamente imposible.** Aquí sí tiene
   sentido (hay árboles gruesos y bajos, y delgados y altos). Si dos predictoras
   están casi perfectamente correlacionadas, la comparación que describe el
   coeficiente **no existe en los datos** (multicolinealidad, sección 15).
3. **Controlar estadísticamente ≠ causalidad.** Ajustar por `Height` no convierte
   a $\beta_1$ en un efecto causal.

---
# 4. Mínimos cuadrados: qué hace el software

## La idea, sin matrices

Entre todos los planos posibles elegimos el que minimiza

$$SCE = \sum_{i=1}^{n}(y_i - \hat y_i)^2$$

¿Por qué los *cuadrados*? Porque (a) penalizan más los errores grandes, (b) evitan
que positivos y negativos se cancelen, y (c) dan solución algebraica cerrada.

## La formulación matricial

$$\mathbf{y} = \mathbf{X}\boldsymbol{\beta} + \boldsymbol{\epsilon}
\qquad\Longrightarrow\qquad
\boxed{\;\hat{\boldsymbol{\beta}} = (\mathbf{X}'\mathbf{X})^{-1}\mathbf{X}'\mathbf{y}\;}$$

siempre que $(\mathbf{X'X})^{-1}$ exista.

## Cálculo paso a paso

In [ ]:
n = len(trees)

# vector y (n x 1)
y = trees["Volume"].to_numpy().reshape(n, 1)
print("y.shape =", y.shape)

# matriz X con la columna de unos (n x p, p = 3)
unos = np.ones((n, 1))
X = np.hstack([unos, trees[["Girth", "Height"]].to_numpy()])
print("X.shape =", X.shape)
print(X[:5])

In [ ]:
XtX     = X.T @ X
XtX_inv = np.linalg.inv(XtX)
Xty     = X.T @ y
beta_hat = XtX_inv @ Xty

print("X'X =\n", XtX.round(2), "\n")
print("(X'X)^-1 =\n", XtX_inv.round(6), "\n")
print("X'y =\n", Xty.round(3), "\n")
print("beta_hat =\n", beta_hat.round(6))

In [ ]:
modelo = smf.ols("Volume ~ Girth + Height", data=trees).fit()

comprobacion = pd.DataFrame({
    "manual":      beta_hat.ravel(),
    "statsmodels": modelo.params.values,
}, index=["Intercepto", "Girth", "Height"])
print(comprobacion)
print("\n¿Coinciden?", np.allclose(beta_hat.ravel(), modelo.params.values))

**Coinciden.** El objetivo no es enseñar álgebra lineal avanzada sino desmitificar
`ols()`: no hay magia, hay una inversa y dos productos matriciales.

In [ ]:
# Matriz sombrero H = X(X'X)^-1 X' : proyecta y sobre el espacio de columnas de X
H = X @ XtX_inv @ X.T
ajustados_manual = (H @ y).ravel()

print(pd.DataFrame({"manual": ajustados_manual[:6],
                    "statsmodels": modelo.fittedvalues.values[:6]}).round(6))

# Los elementos de la diagonal de H son los 'leverages' (sección 16)
print("\nLeverages (primeros 6):", np.diag(H)[:6].round(4))

---
# 5. Ajuste e interpretación del modelo

In [ ]:
print(modelo.summary())

In [ ]:
resumen = pd.DataFrame({
    "coef":      modelo.params,
    "ee":        modelo.bse,
    "t":         modelo.tvalues,
    "p_valor":   modelo.pvalues,
    "IC_2.5%":   modelo.conf_int()[0],
    "IC_97.5%":  modelo.conf_int()[1],
})
resumen.round(5)

## La ecuación estimada

$$\widehat{Volume} = -57.988 + 4.708\,Girth + 0.339\,Height$$

**Intercepto ($\hat\beta_0 = -57.988$).** Volumen promedio de un árbol de 0
pulgadas y 0 pies. **Sin interpretación práctica**: es una extrapolación absurda y
da negativo. Posiciona el plano.

**`Girth` ($\hat\beta_1 = 4.708$).** Entre árboles **de la misma altura**, uno con
una pulgada más de diámetro tiene en promedio **4.71 pies³ más de volumen**.

* Error estándar $= 0.264$ (la incertidumbre de la estimación).
* $t = 17.82$: el coeficiente está a casi 18 errores estándar de cero.
* $p < 10^{-16}$: si el diámetro no aportara nada una vez conocida la altura, ver
  un coeficiente así sería prácticamente imposible.
* **IC 95 %: $[4.167,\ 5.250]$ ← este es el número que se reporta.**

**`Height` ($\hat\beta_2 = 0.339$).** Entre árboles **del mismo diámetro**, uno con
un pie más de altura tiene en promedio **0.34 pies³ más de volumen**.

* $t = 2.607$, $p = 0.0145$: significativo al 5 %, con **mucha menos evidencia**
  que `Girth`.
* IC 95 %: $[0.073,\ 0.606]$. Ancho en términos relativos: sabemos que el efecto
  es positivo, pero su magnitud está mal determinada.

**Error estándar residual $\hat\sigma = 3.882$ pies³** con 28 gl. Como el volumen
promedio es 30.17 pies³, el error típico es $\approx 13\,\%$ del valor medio.

> **Regla de oro.** Nunca reporte solo el signo y el p-valor. Reporte **magnitud +
> unidades + intervalo + la cláusula "manteniendo constante lo demás"**.
> "`Girth` es significativa" no le sirve a un ingeniero forestal; "una pulgada más
> de diámetro, a igual altura, son entre 4.2 y 5.2 pies³ más de madera" sí.

In [ ]:
print(np.sqrt(modelo.mse_resid), "= sigma sombrero (error estándar residual)")
print(modelo.df_resid, "= grados de libertad del error (n - p)")

---
# 6. Significancia global: la prueba $F$

> **Pregunta:** ¿el **conjunto** de predictoras aporta información sobre `Volume`,
> o el modelo no es mejor que predecir siempre la media?

$$H_0:\ \beta_1 = \beta_2 = \cdots = \beta_k = 0
\qquad\text{vs.}\qquad
H_1:\ \text{al menos un } \beta_j \neq 0$$

$$\underbrace{\sum(y_i-\bar y)^2}_{SCT}
=\underbrace{\sum(\hat y_i-\bar y)^2}_{SCR}
+\underbrace{\sum(y_i-\hat y_i)^2}_{SCE}$$

In [ ]:
y_barra = trees["Volume"].mean()
SCT = ((trees["Volume"] - y_barra) ** 2).sum()
SCR = ((modelo.fittedvalues - y_barra) ** 2).sum()
SCE = (modelo.resid ** 2).sum()

k = 2            # número de predictoras
p = k + 1        # número de parámetros

F  = (SCR / k) / (SCE / (n - p))
pv = stats.f.sf(F, k, n - p)

tabla_anova = pd.DataFrame({
    "Fuente":  ["Regresión", "Error", "Total"],
    "SC":      [SCR, SCE, SCT],
    "gl":      [k, n - p, n - 1],
    "CM":      [SCR / k, SCE / (n - p), np.nan],
    "F":       [F, np.nan, np.nan],
    "p_valor": [pv, np.nan, np.nan],
})
tabla_anova

In [ ]:
# Tabla ANOVA con statsmodels: descomposición SECUENCIAL (tipo I), como anova() de R
sm.stats.anova_lm(modelo, typ=1)

In [ ]:
# Comparación explícita contra el modelo nulo = prueba F global
m_nulo = smf.ols("Volume ~ 1", data=trees).fit()
sm.stats.anova_lm(m_nulo, modelo)

**Lectura.** $F = 254.97$ con 2 y 28 gl, $p \approx 1.07\times10^{-18}$. La
variabilidad explicada por el modelo es **255 veces** mayor (por grado de libertad)
que la residual.

> ⚠️ **Ojo con `anova_lm(..., typ=1)`.** Las sumas de cuadrados son
> **secuenciales**: la línea de `Height` mide lo que aporta *después de* `Girth`.
> Si invierte el orden en la fórmula, esos números cambian. Las pruebas $t$ del
> `summary()`, en cambio, son siempre "esta variable dada **todas** las demás".

## Prueba $F$ global ≠ pruebas $t$ individuales

> 🎓 **Pregunta conceptual:** ¿puede el modelo global ser significativo y ninguna
> variable individual serlo?

**Sí, y ocurre con frecuencia.** Son preguntas distintas:

* La $F$ global: *¿aporta algo el conjunto?*
* Cada $t$: *¿aporta algo **esta** variable que las otras no aporten ya?*

Si dos predictoras están muy correlacionadas, ambas pueden ser prescindibles **por
separado** (porque la otra la sustituye) y sin embargo el par ser imprescindible.
La $F$ las evalúa en bloque; las $t$ al margen. También ocurre lo contrario: con
muchas predictoras irrelevantes alguna $t$ puede salir significativa por azar. Por
eso **se mira primero la $F$ y después las $t$**.

In [ ]:
# Con 1 gl en el numerador, F = t^2. Comparar el modelo con y sin Height
# equivale exactamente a la prueba t de Height.
m_solo_girth = smf.ols("Volume ~ Girth", data=trees).fit()
print(sm.stats.anova_lm(m_solo_girth, modelo))
print("\nt de Height al cuadrado:", round(modelo.tvalues['Height'] ** 2, 4))

---
# 7. Calidad del ajuste: $R^2$ y $R^2$ ajustado

$$R^2 = 1-\frac{SCE}{SCT}
\qquad\qquad
R^2_{aj} = 1-\frac{SCE/(n-p)}{SCT/(n-1)}$$

In [ ]:
print("R2         :", round(modelo.rsquared, 6))
print("R2 ajustado:", round(modelo.rsquared_adj, 6))
print("R2 a mano  :", round(1 - SCE / SCT, 6))

$R^2 = 0.948$: el modelo explica el **94.8 %** de la variabilidad del volumen.

## Por qué existe el $R^2$ ajustado

$R^2$ **nunca baja** al agregar predictoras, aunque sean ruido puro. Veámoslo:

In [ ]:
rng = np.random.default_rng(2026)
trees_ruido = trees.copy()
trees_ruido["basura1"] = rng.normal(size=n)
trees_ruido["basura2"] = rng.normal(size=n)
trees_ruido["basura3"] = rng.normal(size=n)

m_basura = smf.ols("Volume ~ Girth + Height + basura1 + basura2 + basura3",
                   data=trees_ruido).fit()

pd.DataFrame({
    "modelo":      ["Girth + Height", "Girth + Height + 3 variables aleatorias"],
    "R2":          [modelo.rsquared,     m_basura.rsquared],
    "R2_ajustado": [modelo.rsquared_adj, m_basura.rsquared_adj],
    "sigma":       [np.sqrt(modelo.mse_resid), np.sqrt(m_basura.mse_resid)],
}).round(5)

$R^2$ **subió** al agregar tres columnas de números aleatorios. El $R^2$ ajustado
penaliza por el número de parámetros y **baja**, que es la respuesta correcta.

> **No use $R^2$ como única medida de calidad.** Un $R^2$ alto es compatible con
> supuestos violados, con una observación influyente que domina el ajuste y con un
> modelo inútil fuera del rango de los datos. El $R^2$ mide cuánto se acerca el
> modelo a **estos** datos, no si el modelo es correcto.

---
# 8. Predicción: dos intervalos que no son lo mismo

In [ ]:
arboles_nuevos = pd.DataFrame({
    "Girth":  [11.0, 13.2, 16.0],
    "Height": [70,   76,   82],
})
arboles_nuevos

In [ ]:
# Predicción puntual
modelo.predict(arboles_nuevos)

In [ ]:
pred = modelo.get_prediction(arboles_nuevos).summary_frame(alpha=0.05)

resultado = pd.concat([arboles_nuevos.reset_index(drop=True), pred], axis=1)
resultado = resultado.rename(columns={
    "mean": "estimacion",
    "mean_ci_lower": "IC_inf", "mean_ci_upper": "IC_sup",     # media
    "obs_ci_lower":  "IP_inf", "obs_ci_upper":  "IP_sup",     # observación nueva
})
resultado[["Girth", "Height", "estimacion", "IC_inf", "IC_sup", "IP_inf", "IP_sup"]].round(3)

## La diferencia conceptual

Ambos intervalos están centrados en la misma predicción puntual, pero responden a
preguntas distintas:

| | Intervalo de **confianza** | Intervalo de **predicción** |
|---|---|---|
| **Pregunta** | ¿Dónde está el volumen **promedio** de *todos* los árboles con $Girth=13.2$, $Height=76$? | ¿Dónde caerá el volumen de **un** árbol concreto con esas medidas? |
| **Objeto estimado** | Un parámetro: $E[Y\mid x_0]$ | Una variable aleatoria: $Y_0$ |
| **Error estándar** | $\hat\sigma\sqrt{\mathbf x_0'(\mathbf{X'X})^{-1}\mathbf x_0}$ | $\hat\sigma\sqrt{1+\mathbf x_0'(\mathbf{X'X})^{-1}\mathbf x_0}$ |
| **Ancho (árbol 2)** | $[28.5,\ 31.4]$ → **2.9** pies³ | $[21.9,\ 38.0]$ → **16.2** pies³ |

El "+1" bajo la raíz es toda la diferencia: al predecir **un individuo** cargamos
además con su variabilidad propia $\sigma^2$, que **no desaparece** aunque
tuviéramos infinitos datos. Por eso el IC de la media se puede angostar aumentando
$n$, mientras que el IP tiene un piso de $\approx \pm 1.96\,\hat\sigma \approx \pm 7.6$
pies³.

> **Error frecuente.** Reportar el IC de la media cuando lo que se quiere es
> predecir un árbol. Si preguntan "¿cuánta madera saco de **este** árbol?", la
> respuesta honesta es el **intervalo de predicción**.

In [ ]:
malla = pd.DataFrame({
    "Girth":  np.linspace(trees.Girth.min(), trees.Girth.max(), 60),
    "Height": trees.Height.mean(),
})
bandas = modelo.get_prediction(malla).summary_frame(alpha=0.05)

fig, ax = plt.subplots()
ax.fill_between(malla.Girth, bandas.obs_ci_lower, bandas.obs_ci_upper,
                color="lightgrey", label="Intervalo de predicción 95%")
ax.fill_between(malla.Girth, bandas.mean_ci_lower, bandas.mean_ci_upper,
                color="steelblue", alpha=.45, label="Intervalo de confianza 95%")
ax.plot(malla.Girth, bandas["mean"], color="firebrick", lw=2, label="Predicción")
ax.scatter(trees.Girth, trees.Volume, color="#333333", s=28, label="Datos")
ax.set(xlabel="Girth (pulgadas)", ylabel="Volume (pies cúbicos)",
       title="Bandas de confianza vs. de predicción (Height fijada en su media)")
ax.legend()
plt.show()

> ⚠️ **Extrapolación.** El modelo solo es creíble dentro del rango observado
> ($Girth \in [8.3, 20.6]$, $Height \in [63, 87]$) **y** en combinaciones plausibles
> de ambas. Predecir un árbol de $Girth = 20$ y $Height = 63$ es extrapolar aunque
> cada valor esté dentro de su rango individual: esa combinación no existe en los
> datos.

---
# 9. Antes de confiar en el modelo: los supuestos

> Obtener coeficientes, $R^2$ y p-valores **no significa que el análisis haya
> terminado**. Significa que apenas empieza la parte interesante.

`ols()` siempre devuelve números. Devuelve coeficientes aunque la relación sea una
parábola, p-valores aunque la varianza explote y $R^2 = 0.99$ aunque un solo dato
esté sosteniendo todo el ajuste. **Los supuestos son las condiciones bajo las
cuales esos números significan lo que creemos que significan.**

| # | Supuesto | Enunciado | Si falla, ¿qué se rompe? |
|---|---|---|---|
| 1 | **Linealidad** (media) | $E[Y \mid \mathbf X] = \beta_0+\beta_1X_1+\cdots+\beta_kX_k$ | **Todo.** Los coeficientes estiman algo que no existe; las predicciones son sesgadas de forma sistemática. |
| 2 | **Homocedasticidad** | $Var(\epsilon_i)=\sigma^2$ constante | Los $\hat\beta$ siguen insesgados, pero **errores estándar, $t$, $F$ e intervalos** son incorrectos. |
| 3 | **Independencia** | $Cov(\epsilon_i,\epsilon_j)=0$ | Igual: la inferencia se vuelve optimista. |
| 4 | **Normalidad** | $\epsilon_i \sim N(0,\sigma^2)$ | Solo la **inferencia exacta en muestras pequeñas**. Con $n$ grande el TLC ayuda. |
| 5 | **Observaciones comparables** | Ninguna domina el ajuste | Las conclusiones dependen de uno o dos datos. Es **robustez**, no distribución. |
| 6 | **Predictoras no colineales** | $(\mathbf{X'X})^{-1}$ bien condicionada | Coeficientes inestables y sin interpretación separable. |

**Jerarquía:** 1 es de vida o muerte; 2 y 3 dañan la inferencia; 4 es la menos
crítica (y la más sobre-enfatizada en los cursos); 5 y 6 no son supuestos sobre
$\epsilon$ sino sobre la **estructura de los datos**.

---
# 10. Residuos: cuatro versiones y por qué hacen falta

**Residuo ordinario:** $e_i = y_i - \hat y_i$.

Problema: **no tienen varianza constante aunque el modelo sea perfecto**. Como
$\mathbf e = (\mathbf I - \mathbf H)\boldsymbol\epsilon$,

$$Var(e_i) = \sigma^2\,(1 - h_{ii})$$

donde $h_{ii}$ es el **leverage**. Las observaciones con predictoras extremas
tienen $h_{ii}$ alto y, por construcción, residuos **más pequeños**: el modelo se
"estira" para alcanzarlas.

**Estandarizado (con $\sigma$ conocido):** $z_i = e_i / (\sigma\sqrt{1-h_{ii}})$.
Correcto en teoría, inútil en la práctica.

**Studentizado internamente** (`resid_studentized_internal`):
$r_i = e_i / (\hat\sigma\sqrt{1-h_{ii}})$. Si la observación $i$ es atípica,
**infla su propio $\hat\sigma$** y se disimula a sí misma.

**Studentizado externamente** (`resid_studentized_external`):
$r_i^{*} = e_i / (\hat\sigma_{(i)}\sqrt{1-h_{ii}})$, con $\hat\sigma_{(i)}$
estimado **sin** la observación $i$. Numerador y denominador independientes, y
$r_i^{*}\sim t_{n-p-1}$. **Este es el que se usa para detectar atípicos.**

> **En una frase:** los residuos studentizados ponen a todas las observaciones en
> la misma escala, corrigiendo el hecho de que el modelo se ajusta más a unos
> puntos que a otros.

In [ ]:
infl = modelo.get_influence()

diagnostico = pd.DataFrame({
    "Girth":           trees["Girth"].values,
    "Height":          trees["Height"].values,
    "observado":       trees["Volume"].values,
    "estimado":        modelo.fittedvalues.values,
    "residual":        modelo.resid.values,
    "r_estandarizado": infl.resid_studentized_internal,   # = rstandard() de R
    "r_studentizado":  infl.resid_studentized_external,   # = rstudent()  de R
    "leverage":        infl.hat_matrix_diag,              # = hatvalues() de R
    "cook":            infl.cooks_distance[0],            # = cooks.distance() de R
}, index=trees.index)

diagnostico.round(4)

In [ ]:
umbral_h    = 2 * p / n        # leverage alto
umbral_cook = 4 / (n - p)      # influencia (una de varias reglas)

print(f"umbral leverage 2p/n = {umbral_h:.4f}")
print(f"umbral Cook 4/(n-p)  = {umbral_cook:.4f}")
print(f"umbral residuo       = 2 (en valor absoluto)")

In [ ]:
marcadas = diagnostico[(diagnostico.r_studentizado.abs() > 2) |
                       (diagnostico.leverage > umbral_h) |
                       (diagnostico.cook > umbral_cook)]
marcadas[["Girth", "Height", "observado", "estimado",
          "r_studentizado", "leverage", "cook"]].round(4)

Cinco observaciones se levantan la mano. La **31** aparece en las tres listas: es
el árbol grande que vimos en el boxplot. Volvemos a ella en la sección 16.

---
# 11. Supuesto de linealidad

**¿Qué esperamos ver?** Una nube **sin forma**: residuos repartidos al azar
alrededor de cero, con la misma dispersión a lo largo de todo el eje horizontal.
Cualquier patrón sistemático es información que el modelo **no capturó**.

In [ ]:
def suaviza(x, y, ax, color="darkorange"):
    '''Curva suavizada (lowess) para ver el patrón medio de los residuos.'''
    lo = sm.nonparametric.lowess(y, x, frac=0.8)
    ax.plot(lo[:, 0], lo[:, 1], color=color, lw=2)

r_stud = infl.resid_studentized_external

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, (xx, nombre) in zip(axes, [(modelo.fittedvalues, "Valores ajustados"),
                                   (trees.Girth,  "Girth"),
                                   (trees.Height, "Height")]):
    ax.scatter(xx, r_stud, s=40, color="steelblue")
    ax.axhline(0, color="grey")
    ax.axhline( 2, ls="--", color="firebrick")
    ax.axhline(-2, ls="--", color="firebrick")
    suaviza(np.asarray(xx), r_stud, ax)
    ax.set(xlabel=nombre, ylabel="Residuo studentizado", title=f"Residuos vs. {nombre}")
plt.tight_layout(); plt.show()

**Diagnóstico.** Los gráficos contra los ajustados y contra `Girth` muestran una
**curvatura en U clarísima**: residuos positivos en los extremos, negativos en el
centro. No es ruido: es una **señal**.

In [ ]:
# Tres patrones que hay que aprender a reconocer
rng2 = np.random.default_rng(1)
xx = np.linspace(0, 10, 120)
paneles = [
    ("A) Correcto: nube sin forma",               rng2.normal(0, 1, 120)),
    ("B) Curvatura: falta un término no lineal",  1.2*(xx-5)**2/5 - 3 + rng2.normal(0, .6, 120)),
    ("C) Embudo: heterocedasticidad",             rng2.normal(0, 1, 120) * (.25 + xx/4)),
]
fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
for ax, (titulo, ee) in zip(axes, paneles):
    ax.scatter(xx, ee, s=12, color="steelblue")
    ax.axhline(0, color="firebrick")
    ax.set(title=titulo, xlabel="Valores ajustados", ylabel="Residuos")
plt.tight_layout(); plt.show()

Un patrón de curvatura puede significar tres cosas:

1. **Falta una transformación** de alguna predictora ($\log X$, $\sqrt X$, $X^2$).
2. **Falta una variable** en el modelo (variable omitida).
3. **La forma funcional es otra** (multiplicativa en lugar de aditiva).

En `trees` sabemos cuál es: el volumen de un sólido no es una función lineal de sus
dimensiones. Lo resolvemos en la sección 18.

## Gráficos de regresión parcial

Aíslan la relación de cada predictora con $Y$ una vez descontadas las demás. Son la
herramienta correcta para decidir **cuál** predictora necesita transformarse.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, v in zip(axes, ["Girth", "Height"]):
    otra = "Height" if v == "Girth" else "Girth"
    rx = smf.ols(f"{v} ~ {otra}",    data=trees).fit().resid
    ry = smf.ols(f"Volume ~ {otra}", data=trees).fit().resid
    ax.scatter(rx, ry, s=40, color="steelblue")
    b = np.polyfit(rx, ry, 1)
    xs = np.linspace(rx.min(), rx.max(), 50)
    ax.plot(xs, np.polyval(b, xs), color="firebrick", lw=2)
    ax.axhline(0, ls=":", color="grey"); ax.axvline(0, ls=":", color="grey")
    ax.set(title=f"Regresión parcial: {v} | {otra}",
           xlabel=f"{v} ajustado por {otra}", ylabel=f"Volume ajustado por {otra}")
plt.tight_layout(); plt.show()

---
# 12. Normalidad de los residuos

$$H_0:\ \text{los residuos son compatibles con una distribución normal}$$
$$H_1:\ \text{los residuos no son compatibles con la normalidad}$$

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))

axes[0].hist(r_stud, bins=8, density=True, color="steelblue", edgecolor="white")
xs = np.linspace(-3.5, 3.5, 200)
dens = stats.gaussian_kde(r_stud)
axes[0].plot(xs, dens(xs), color="darkorange", lw=2, label="Densidad residuos")
axes[0].plot(xs, stats.norm.pdf(xs), ls="--", color="firebrick", lw=2, label="N(0,1)")
axes[0].set(title="Histograma y densidad", xlabel="Residuo studentizado")
axes[0].legend(fontsize=8)

sm.qqplot(r_stud, line="45", fit=True, ax=axes[1],
          markerfacecolor="steelblue", markeredgecolor="steelblue", alpha=.8)
axes[1].set_title("Gráfico Q-Q normal")
plt.tight_layout(); plt.show()

In [ ]:
W, p_shapiro = stats.shapiro(r_stud)
print(f"Shapiro-Wilk sobre los residuos studentizados: W = {W:.4f}, p = {p_shapiro:.4f}")

**Lectura.** $W = 0.973$, $p = 0.612$. No hay evidencia contra la normalidad. En el
Q-Q plot los puntos siguen la línea salvo en el extremo superior derecho (la
observación 31, otra vez).

## Tres advertencias sobre este supuesto

1. **El p-valor no sustituye al gráfico.** Shapiro-Wilk resume en un número lo que
   el Q-Q muestra en detalle. Aquí el test "pasa" pero el Q-Q señala un punto que el
   test no captura. **Mire siempre el gráfico.**
2. **Con $n$ grande, los tests detectan desviaciones triviales.** Con $n = 5000$
   Shapiro-Wilk rechaza por asimetrías sin ninguna relevancia práctica. El test
   responde "¿es *exactamente* normal?", y con datos reales la respuesta siempre es no.
3. **La normalidad no es un requisito para "poder hacer regresión".** Mínimos
   cuadrados es el mejor estimador lineal insesgado (Gauss-Markov) **sin** suponer
   normalidad. Se necesita para que las pruebas $t$/$F$ y los intervalos sean
   *exactos* en muestras pequeñas. Con $n = 31$ importa; con $n = 3000$ es casi
   irrelevante.

> **Y lo más importante:** el supuesto es sobre los **errores**, no sobre $Y$ ni
> sobre las predictoras. Que `Volume` sea asimétrica no viola nada por sí mismo.

---
# 13. Homocedasticidad

$$Var(\epsilon_i) = \sigma^2 \quad \text{para todo } i$$

In [ ]:
r_int = infl.resid_studentized_internal

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))

axes[0].scatter(modelo.fittedvalues, r_stud, s=40, color="steelblue")
axes[0].axhline(0, color="grey")
axes[0].axhline(2, ls="--", color="firebrick"); axes[0].axhline(-2, ls="--", color="firebrick")
axes[0].set(title="Residuos vs. ajustados", xlabel="Ajustados", ylabel="Residuo studentizado")

axes[1].scatter(modelo.fittedvalues, np.sqrt(np.abs(r_int)), s=40, color="steelblue")
suaviza(modelo.fittedvalues.values, np.sqrt(np.abs(r_int)), axes[1])
axes[1].set(title="Scale-Location", xlabel="Ajustados", ylabel=r"$\sqrt{|r_{estandarizado}|}$")
plt.tight_layout(); plt.show()

El gráfico **Scale-Location** usa $\sqrt{|r_i|}$ porque elimina el signo (interesa
la *magnitud*) y estabiliza la escala. **Queremos una línea horizontal** con puntos
repartidos por igual.

## Prueba de Breusch-Pagan

$$H_0:\ \text{varianza constante} \qquad H_1:\ \text{la varianza depende de los predictores}$$

La lógica: si la varianza fuese constante, los residuos al cuadrado no deberían
poder predecirse a partir de las $X$.

In [ ]:
bp = het_breuschpagan(modelo.resid, modelo.model.exog)
print(pd.Series(bp, index=["LM", "p-valor LM", "F", "p-valor F"]).round(4))

**Lectura.** $LM = 2.468$, gl $= 2$, $p = 0.291$. **No rechazamos** la
homocedasticidad. Ojo: con $n = 31$ la prueba tiene poca potencia; "no rechazar" no
es "confirmar".

> **Nota R ↔ Python.** `het_breuschpagan()` de `statsmodels` devuelve la versión
> **studentizada (Koenker)**, que es exactamente la que calcula `lmtest::bptest()`
> de R por defecto. **Coinciden.** Con `bptest(m, studentize = FALSE)` obtendría
> $1.7375$: la versión original, que es otra prueba.

In [ ]:
# Cómo se ve un EMBUDO (datos simulados con varianza creciente)
rng3 = np.random.default_rng(7)
xe = rng3.uniform(1, 10, 200)
ye = 2 + 1.5*xe + rng3.normal(0, .4*xe)
me = sm.OLS(ye, sm.add_constant(xe)).fit()

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
axes[0].scatter(xe, ye, s=14, color="grey")
axes[0].plot(np.sort(xe), me.params[0] + me.params[1]*np.sort(xe), color="firebrick", lw=2)
axes[0].set(title="Datos con varianza creciente", xlabel="x", ylabel="y")
axes[1].scatter(me.fittedvalues, me.get_influence().resid_studentized_internal,
                s=14, color="steelblue")
axes[1].axhline(0, color="grey")
axes[1].set(title="Patrón de EMBUDO", xlabel="Ajustados", ylabel="Residuo estandarizado")
plt.tight_layout(); plt.show()

La dispersión crece con el nivel: típico cuando $Y$ es un conteo, un monto, una
concentración o cualquier cantidad positiva con efecto multiplicativo. La solución
habitual es transformar $Y$ (log, raíz) o usar errores estándar robustos.

---
# 14. Independencia

$$H_0:\ \rho_1 = 0\ \ (\text{ausencia de autocorrelación de orden 1})$$

$$d = \frac{\sum_{t=2}^{n}(e_t - e_{t-1})^2}{\sum_{t=1}^{n}e_t^2}$$

$d\approx 2$ sin autocorrelación, $d\approx 0$ positiva, $d\approx 4$ negativa.

In [ ]:
print("Durbin-Watson =", round(durbin_watson(modelo.resid), 4))
# statsmodels devuelve SOLO el estadístico; lmtest::dwtest() de R da también el p-valor.

## ⚠️ Por qué aquí este resultado NO se debe interpretar

$d = 1.27$ sugeriría autocorrelación positiva. **Pero Durbin-Watson depende del
orden de las filas**, y en `trees` ese orden **no tiene significado sustantivo**: el
archivo está ordenado por `Girth` creciente. Lo que DW detecta es la **curvatura**
de la sección 11 —residuos consecutivos parecidos porque siguen una U— y no una
dependencia temporal.

In [ ]:
trees_barajado = trees.sample(frac=1, random_state=99)
m_barajado = smf.ols("Volume ~ Girth + Height", data=trees_barajado).fit()

print("DW con el orden original  :", round(durbin_watson(modelo.resid), 4))
print("DW con el orden aleatorio :", round(durbin_watson(m_barajado.resid), 4))
print("\n¿Es el mismo modelo?", np.allclose(modelo.params, m_barajado.params))

Reordenando las filas al azar el estadístico cambia por completo, y el modelo es
**exactamente el mismo**. Eso demuestra que aquí DW no mide una propiedad del
modelo sino un artefacto del orden del archivo.

**Cuándo SÍ tiene sentido:** series de tiempo, mediciones repetidas sobre el mismo
sujeto, datos recogidos secuencialmente por un mismo operario o instrumento, o
cualquier situación con un orden natural y real. En un corte transversal de árboles
talados, no.

> **Regla:** aplique DW solo si puede responder *"¿qué significa que la observación
> $i$ venga antes que la $i+1$?"*. Si no hay respuesta, no aplique la prueba.

---
# 15. Multicolinealidad

En regresión simple no existe. En regresión múltiple es un problema central.

## Qué es y por qué aparece

**Multicolinealidad** = las predictoras están (fuertemente) correlacionadas entre
sí. Aparece porque miden aspectos relacionados del mismo fenómeno (peso y talla;
ingreso y educación; diámetro y altura), porque una es combinación casi exacta de
otras, o porque el diseño del estudio no separó las variables.

## Por qué produce coeficientes inestables

$$Var(\hat\beta_j) = \frac{\sigma^2}{\sum(x_{ij}-\bar x_j)^2}\cdot
\underbrace{\frac{1}{1-R_j^2}}_{VIF_j}$$

donde $R_j^2$ es el $R^2$ de regresar $X_j$ contra **las demás predictoras**. Si
$R_j^2 \to 1$, el VIF explota y el error estándar con él. Intuitivamente: el
coeficiente parcial de $X_j$ se estima con **la parte de $X_j$ que las otras no
explican**; si esa parte es minúscula, se estima con casi nada de información.

In [ ]:
X_vif = sm.add_constant(trees[["Girth", "Height"]])

vif = pd.DataFrame({
    "variable": X_vif.columns,
    "VIF": [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])],
})
print(vif.round(4))

# Con dos predictoras, VIF = 1/(1 - r^2)
r_gh = trees.Girth.corr(trees.Height)
print(f"\nr(Girth, Height) = {r_gh:.4f}   ->   VIF = 1/(1-r^2) = {1/(1-r_gh**2):.4f}")

**Lectura.** $VIF = 1.37$ para ambas: `Height` explica solo el 27 % de la
variabilidad de `Girth`. Sobra información independiente. **No hay problema de
multicolinealidad.**

## Cómo interpretar el VIF (y cómo no)

Se citan umbrales de 5 o de 10. **No son leyes.**

* $\sqrt{VIF}$ dice **cuántas veces más ancho** es el error estándar por la
  colinealidad. Con $VIF = 1.37$, $\sqrt{1.37} = 1.17$: un 17 % más ancho,
  irrelevante. Con $VIF = 10$ es 3.2 veces más ancho: eso sí se nota.
* Un VIF alto **no invalida el modelo** si el objetivo es **predecir**: las
  predicciones y sus intervalos siguen siendo correctos. Daña la **interpretación
  de coeficientes individuales**.
* Un VIF de 8 con errores estándar pequeños ($n$ grande) puede ser tolerable; un
  VIF de 4 con $n=20$ puede arruinar el análisis. **El VIF se lee junto al error
  estándar, no solo.**
* Las *dummies* de un factor con muchos niveles y los términos polinómicos o de
  interacción **siempre** tienen VIF alto por construcción. Eso es estructural.

---
# 16. Outlier, leverage e influencia: tres cosas distintas

> $$\text{outlier} \neq \text{leverage} \neq \text{influencia}$$

| Concepto | Qué es | Se mide con | Dónde es raro |
|---|---|---|---|
| **Outlier** | Valor inusual **en la respuesta**, dado el modelo. | $\lvert r_i^{*}\rvert$ grande | Eje $Y$ |
| **Leverage alto** | Valores inusuales **en las predictoras**. | $h_{ii}$ grande | Eje(s) $X$ |
| **Influyente** | Su presencia **cambia el modelo**. Suele requerir *ambas* cosas. | Distancia de Cook | Efecto sobre $\hat\beta$ |

Intuición geométrica: un punto con leverage alto es un niño en el extremo de un
balancín. Si además está fuera de la recta (outlier), mueve el balancín entero. Si
está **sobre** la recta, tiene mucho leverage y **cero influencia**.

$$D_i = \frac{r_i^2}{p}\cdot\frac{h_{ii}}{1-h_{ii}}
\qquad \leftarrow \text{residuo} \times \text{leverage}$$

La fórmula **es** la afirmación: influencia = ser atípico **y** estar en una
posición que importa.

In [ ]:
lev  = infl.hat_matrix_diag
cook = infl.cooks_distance[0]
idx  = np.arange(1, n + 1)

fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))

axes[0].scatter(idx, r_stud, s=40, color="steelblue")
axes[0].axhline(0, color="grey")
axes[0].axhline(2, ls="--", color="firebrick"); axes[0].axhline(-2, ls="--", color="firebrick")
for i in idx[np.abs(r_stud) > 2]:
    axes[0].annotate(str(i), (i, r_stud[i-1]), textcoords="offset points",
                     xytext=(6, 0), color="firebrick")
axes[0].set(title="Outliers en la respuesta", xlabel="Observación",
            ylabel="Residuo studentizado", ylim=(-3.5, 3.5))

axes[1].scatter(idx, lev, s=40, color="steelblue")
axes[1].axhline(umbral_h, ls="--", color="firebrick")
for i in idx[lev > umbral_h]:
    axes[1].annotate(str(i), (i, lev[i-1]), textcoords="offset points",
                     xytext=(-16, 0), color="firebrick")
axes[1].set(title="Leverage (h)", xlabel="Observación", ylabel="h")

axes[2].vlines(idx, 0, cook, color="steelblue", lw=2)
axes[2].axhline(umbral_cook, ls="--", color="firebrick")
axes[2].annotate(str(idx[np.argmax(cook)]), (idx[np.argmax(cook)], cook.max()),
                 textcoords="offset points", xytext=(-18, -4), color="firebrick")
axes[2].set(title="Distancia de Cook", xlabel="Observación", ylabel="D")

plt.tight_layout(); plt.show()

In [ ]:
# Residuals vs Leverage, con contornos de distancia de Cook (equivale a plot(modelo, 5) de R)
fig, ax = plt.subplots(figsize=(7, 4.8))
ax.scatter(lev, r_int, s=45, color="steelblue")
ax.axhline(0, color="grey")

hs = np.linspace(0.001, lev.max() * 1.15, 200)
for D, estilo in [(0.5, "--"), (1.0, ":")]:
    contorno = np.sqrt(D * p * (1 - hs) / hs)
    ax.plot(hs,  contorno, ls=estilo, color="firebrick", lw=1.2)
    ax.plot(hs, -contorno, ls=estilo, color="firebrick", lw=1.2)
    ax.annotate(f"Cook = {D}", (hs[-1], contorno[-1]), color="firebrick", fontsize=8)

for i in idx[(cook > umbral_cook) | (lev > umbral_h)]:
    ax.annotate(str(i), (lev[i-1], r_int[i-1]), textcoords="offset points",
                xytext=(6, 4), color="firebrick")

ax.set(xlabel="Leverage", ylabel="Residuo estandarizado",
       title="Residuals vs Leverage", ylim=(-3.2, 3.6), xlim=(0, lev.max()*1.2))
plt.show()

Lo preocupante es la **esquina superior o inferior derecha**: residuo grande *y*
leverage alto.

In [ ]:
tabla = diagnostico.assign(
    outlier    = diagnostico.r_studentizado.abs() > 2,
    lev_alto   = diagnostico.leverage > umbral_h,
    influyente = diagnostico.cook > umbral_cook,
)
tabla[tabla.outlier | tabla.lev_alto | tabla.influyente][
    ["Girth", "Height", "observado", "estimado", "r_studentizado",
     "leverage", "cook", "outlier", "lev_alto", "influyente"]].round(4)

**Lectura de la tabla:**

* **Obs. 20** ($Girth=13.8$, $Height=64$): leverage alto ($h = 0.211$) pero residuo
  pequeño ($r^* = -1.11$) y Cook bajo. Un árbol con combinación rara de predictoras
  —grueso y **bajo**— que el modelo predice bien. **Leverage sin influencia.**
* **Obs. 2, 3 y 18:** Cook por encima del umbral pero residuos y leverage moderados.
  Casos limítrofes.
* **Obs. 31** ($Girth=20.6$, $Height=87$, $Volume=77$): el árbol más grande.
  $h = 0.227$ (el mayor), $r^* = 2.77$ (el mayor) y $D = 0.605$, **tres veces** el
  segundo valor más alto. **Este es el caso serio.**

In [ ]:
reglas = {
    "D > 4/(n-p)":    4 / (n - p),
    "D > 4/n":        4 / n,
    "D > 1":          1.0,
    "D > 3*media(D)": 3 * cook.mean(),
}
pd.DataFrame({
    "regla":  list(reglas),
    "umbral": list(reglas.values()),
    "marcadas": [", ".join(map(str, idx[cook > u])) or "(ninguna)" for u in reglas.values()],
})

Según la regla elegida, "las observaciones influyentes" son cuatro, cuatro, ninguna
o una. **Ningún umbral es la verdad.** Lo informativo es la **separación
relativa**: la observación 31 está sola, muy por encima de todas, con cualquier
criterio.

---
# 17. Análisis de sensibilidad

La pregunta no es *"¿hay observaciones influyentes?"* sino **"¿cambia mi conclusión
si no estuvieran?"**.

In [ ]:
trees_sin31 = trees.drop(index=31)
modelo_sin31 = smf.ols("Volume ~ Girth + Height", data=trees_sin31).fit()

def resumir(m):
    return pd.Series({
        "Intercepto":  m.params["Intercept"],
        "Girth":       m.params["Girth"],
        "ee_Girth":    m.bse["Girth"],
        "p_Girth":     m.pvalues["Girth"],
        "Height":      m.params["Height"],
        "ee_Height":   m.bse["Height"],
        "p_Height":    m.pvalues["Height"],
        "R2_ajustado": m.rsquared_adj,
        "sigma":       np.sqrt(m.mse_resid),
    })

comp = pd.DataFrame({"Con obs. 31": resumir(modelo),
                     "Sin obs. 31": resumir(modelo_sin31)})
comp["Cambio %"] = 100 * (comp["Sin obs. 31"] - comp["Con obs. 31"]) / comp["Con obs. 31"].abs()
comp.round(4)

In [ ]:
pd.DataFrame({
    "Girth":     arboles_nuevos.Girth,
    "Height":    arboles_nuevos.Height,
    "con_obs31": modelo.predict(arboles_nuevos).values,
    "sin_obs31": modelo_sin31.predict(arboles_nuevos).values,
}).assign(diferencia=lambda d: d.sin_obs31 - d.con_obs31).round(3)

## ¿Cambió sustancialmente la conclusión?

**No.** Al quitar la observación 31:

* `Girth` baja de $4.708$ a $4.478$ (−4.9 %), y el nuevo valor sigue **dentro del
  IC 95 % del modelo original** ($[4.167, 5.250]$).
* `Height` baja de $0.339$ a $0.299$ (−11.8 %) y sigue siendo significativo
  ($p = 0.017$ frente a $p = 0.015$).
* $R^2$ ajustado baja levemente ($0.9442 \to 0.9395$) y $\hat\sigma$ mejora
  ($3.882 \to 3.490$), lo esperable al retirar el punto peor predicho.
* Las predicciones se mueven **menos de 2 pies³** en el rango típico.

**Conclusión:** la observación 31 es el punto que más tira del modelo, pero **el
modelo no depende de ella**. Ninguna decisión cambiaría. Por lo tanto **la
conservamos**.

## ⚠️ Lo que NO se debe hacer

> **Nunca elimine una observación automáticamente porque tenga Cook alto, leverage
> alto o residuo grande.** Esos estadísticos son **señales para investigar**, no
> veredictos.

Antes de siquiera considerar una exclusión:

1. **¿Error de digitación?** ¿Volumen 77 o 7.7? → se corrige, no se borra.
2. **¿Error de medición?** → se documenta y se corrige o se excluye **con
   justificación registrada**.
3. **¿Caso válido pero excepcional?** Este árbol de 20.6 pulgadas es real: es el
   más grande de la muestra. → **Se conserva.** Suele ser la información más valiosa.
4. **¿Pertenece a otra población?** → se separa el análisis, no se borra.
5. **¿La estructura del modelo es insuficiente?** Si un solo punto "no encaja",
   quizá el problema no es el punto. → **Este es justamente nuestro caso.**

Y si tras todo eso se excluye: **se reporta**, se explica por qué y se muestran los
resultados **con y sin** ella.

---
# 18. ¿Qué hacer cuando no se cumplen los supuestos?

| Problema | Cómo detectarlo | Posibles causas | Posibles tratamientos |
|---|---|---|---|
| **No linealidad** | Residuos vs. ajustados con curvatura; gráficos parciales con forma | Forma funcional incorrecta; variable omitida; relación multiplicativa | Transformar $X$ ($\log X$, $\sqrt X$, $X^2$); términos polinómicos; transformar $Y$; incluir la variable faltante; GAM/splines |
| **Heterocedasticidad** | Embudo; Scale-Location creciente; Breusch-Pagan significativo | $Y$ es conteo/monto/concentración; efecto multiplicativo | Transformar $Y$; **errores estándar robustos (HC)**; mínimos cuadrados ponderados; GLM |
| **No normalidad** | Q-Q con colas desviadas; Shapiro-Wilk significativo | Outliers; asimetría; modelo mal especificado; $Y$ discreta | Revisar outliers primero; transformar; considerar $n$ (TLC); revisar especificación; GLM |
| **Dependencia** | DW lejos de 2 **con orden con sentido**; ACF de residuos | Serie de tiempo; medidas repetidas; conglomerados | Estructura de correlación (AR); errores estándar por conglomerado; modelos mixtos |
| **Influencia** | Cook alto; leverage alto; Residuals vs Leverage | Error de datos; caso excepcional; otra población; modelo insuficiente | **Verificar el dato**; sensibilidad; comparar con y sin; regresión robusta; justificar toda exclusión |
| **Multicolinealidad** | VIF alto; correlaciones altas; signos "imposibles"; EE enormes con $R^2$ alto | Variables redundantes; muestra pequeña | Conocimiento del dominio; retirar una redundante **con justificación**; combinar en un índice; centrar; regularización **como extensión** |

## Aplicándolo a `trees`: tratamos la no linealidad

Nuestro diagnóstico detectó **un** problema real: curvatura. Y tenemos una teoría
sobre su origen. Un árbol se parece a un cilindro o a un cono:

$$V = \pi\left(\frac{d}{2}\right)^2 h \cdot c
\qquad\Longrightarrow\qquad V \propto d^2\,h$$

Esto **no** es aditivo, es **multiplicativo**. Y un modelo multiplicativo se vuelve
lineal tomando logaritmos:

$$\log V = \log c' + 2\log d + 1\log h + \epsilon$$

**Predicción teórica antes de ajustar:** el coeficiente de $\log(Girth)$ debería
salir cerca de **2** y el de $\log(Height)$ cerca de **1**.

In [ ]:
modelo_log = smf.ols("np.log(Volume) ~ np.log(Girth) + np.log(Height)", data=trees).fit()
print(modelo_log.summary())

In [ ]:
ic = modelo_log.conf_int().rename(columns={0: "IC 2.5%", 1: "IC 97.5%"})
ic["estimacion"] = modelo_log.params
ic["valor_teorico"] = [np.nan, 2.0, 1.0]
ic[["estimacion", "IC 2.5%", "IC 97.5%", "valor_teorico"]].round(4)

**El resultado es notable:**

| Coeficiente | Estimación | IC 95 % | Valor teórico | ¿Compatible? |
|---|---|---|---|---|
| $\log(Girth)$ | $1.983$ | $[1.829,\ 2.136]$ | $2$ | ✅ el IC contiene 2 |
| $\log(Height)$ | $1.117$ | $[0.698,\ 1.536]$ | $1$ | ✅ el IC contiene 1 |

Los datos **recuperan la geometría del sólido**. Esto es lo que se ve cuando un
modelo está bien especificado: los coeficientes dejan de ser números arbitrarios y
adquieren significado sustantivo.

## Volver a diagnosticar

In [ ]:
infl_log = modelo_log.get_influence()
r_log    = infl_log.resid_studentized_external

fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))

axes[0].scatter(modelo_log.fittedvalues, r_log, s=40, color="steelblue")
axes[0].axhline(0, color="grey")
suaviza(modelo_log.fittedvalues.values, r_log, axes[0])
axes[0].set(title="Residuos vs. ajustados", xlabel="Ajustados (escala log)",
            ylabel="Residuo studentizado")

sm.qqplot(r_log, line="45", fit=True, ax=axes[1],
          markerfacecolor="steelblue", markeredgecolor="steelblue", alpha=.8)
axes[1].set_title("Q-Q normal")

axes[2].vlines(idx, 0, infl_log.cooks_distance[0], color="steelblue", lw=2)
axes[2].axhline(umbral_cook, ls="--", color="firebrick")
axes[2].set(title="Distancia de Cook", xlabel="Observación", ylabel="D")

plt.tight_layout(); plt.show()

In [ ]:
print("Shapiro-Wilk :", [round(v, 4) for v in stats.shapiro(r_log)])
print("Breusch-Pagan:", [round(v, 4) for v in het_breuschpagan(modelo_log.resid, modelo_log.model.exog)[:2]])
ck_log = infl_log.cooks_distance[0]
print(f"Cook máximo  : {ck_log.max():.4f} en la observación {idx[np.argmax(ck_log)]}")
print(f"(en el modelo lineal era {cook.max():.4f} en la observación {idx[np.argmax(cook)]})")

**La curvatura desapareció.** Los residuos vs. ajustados ahora son una nube sin
forma. Normalidad ($p = 0.278$) y homocedasticidad ($p = 0.256$) sin problemas. Y
la distancia de Cook máxima bajó de $0.605$ (obs. 31) a $0.224$ (obs. 18): **el
punto influyente dejó de serlo**.

> 🎯 **Esta es la lección más importante de la clase.** La observación 31 no era un
> dato malo. Era el dato que más claramente delataba que el **modelo** estaba mal
> especificado. Al corregir la forma funcional, el "problema" se disolvió solo.
> **Antes de sospechar del dato, sospeche del modelo.**

## Interpretación del modelo log-log

En un modelo log-log los coeficientes son **elasticidades**:

$$\hat\beta_j = \frac{\%\,\text{de cambio en } Y}{\%\,\text{de cambio en } X_j}$$

* **$\log(Girth) = 1.983$:** a igual altura, un **1 % más de diámetro** se asocia
  con un **1.98 % más de volumen**. (Un 10 % más de diámetro ≈ 21 % más de volumen:
  $1.10^{1.983} = 1.21$.)
* **$\log(Height) = 1.117$:** a igual diámetro, un 1 % más de altura ≈ 1.12 % más
  de volumen.

In [ ]:
# Comparación en la ESCALA ORIGINAL (única comparación justa entre los dos modelos)
pred_lineal = modelo.fittedvalues
pred_log    = np.exp(modelo_log.fittedvalues)      # se deshace la transformación

pd.DataFrame({
    "modelo": ["Lineal: Volume ~ Girth + Height", "Log-log: log(V) ~ log(G) + log(H)"],
    "RMSE_pies3": [np.sqrt(np.mean((trees.Volume - pred_lineal) ** 2)),
                   np.sqrt(np.mean((trees.Volume - pred_log) ** 2))],
    "EAM_pies3":  [np.mean(np.abs(trees.Volume - pred_lineal)),
                   np.mean(np.abs(trees.Volume - pred_log))],
}).round(4)

El error de predicción baja de **3.69** a **2.42** pies³: una mejora del 35 %.

> ⚠️ **No compare los $R^2$ de estos dos modelos.** Uno explica la variabilidad de
> `Volume` y el otro la de `log(Volume)`: son variables respuesta distintas y los
> $R^2$ **no son comparables**. La comparación válida es en la escala original.

## El ciclo completo

$$\textbf{MODELO} \rightarrow \textbf{DIAGNÓSTICO} \rightarrow \textbf{TRATAMIENTO} \rightarrow \textbf{NUEVO MODELO} \rightarrow \textbf{NUEVO DIAGNÓSTICO}$$

1. **Modelo:** `Volume ~ Girth + Height`. $R^2 = 0.948$, todo significativo.
2. **Diagnóstico:** curvatura en los residuos; obs. 31 influyente.
3. **Tratamiento:** teoría del sólido → transformación logarítmica.
4. **Nuevo modelo:** coeficientes ≈ 2 y ≈ 1, coherentes con la geometría.
5. **Nuevo diagnóstico:** residuos sin patrón, supuestos en orden, sin puntos
   influyentes, RMSE 35 % menor. **Aquí paramos.**

---
# 19. Variables categóricas como predictoras

Hasta ahora todas las predictoras eran numéricas. ¿Qué pasa si una es una categoría
(sexo, región, tipo de suelo, presencia de piscina)?

Conjunto pequeño y **didáctico** (construido para esta clase) sobre precios de
vivienda: `Precio` en millones, `Area` en m², `Piscina` con tres niveles.

In [ ]:
viviendas = pd.DataFrame({
    "Precio":  [105, 112, 134, 137, 141, 164, 166, 176, 175, 191, 190, 200, 245, 227, 270],
    "Area":    [ 60,  72,  85,  78,  95, 102,  88, 110, 125, 105, 130, 140, 148, 160, 170],
    "Piscina": ["Sin","Sin","Sin","Pequena","Sin","Pequena","Grande","Pequena",
                "Sin","Grande","Pequena","Sin","Grande","Pequena","Grande"],
})
print(viviendas.Piscina.value_counts())
viviendas.head()

## Por qué NO se codifica 1, 2, 3

La tentación es reemplazar `Sin → 1`, `Pequena → 2`, `Grande → 3` y meterla como
numérica. **Eso impone dos supuestos falsos:**

1. Que las categorías están **ordenadas** (¿lo están? aquí quizá; ¿y con "Bogotá,
   Medellín, Cali"?).
2. Que los **saltos son iguales**: pasar de *Sin* a *Pequeña* valdría exactamente lo
   mismo que pasar de *Pequeña* a *Grande*. Eso es una restricción impuesta por el
   analista, no por los datos.

In [ ]:
viviendas["Cod"] = viviendas.Piscina.map({"Sin": 1, "Pequena": 2, "Grande": 3})  # NO HACER

m_malo  = smf.ols("Precio ~ Area + Cod", data=viviendas).fit()
m_bueno = smf.ols('Precio ~ Area + C(Piscina, Treatment(reference="Sin"))',
                  data=viviendas).fit()

pd.DataFrame({
    "enfoque":     ["Codificación 1-2-3 (incorrecta)", "Variables dummy (correcta)"],
    "parametros":  [len(m_malo.params), len(m_bueno.params)],
    "sigma":       [np.sqrt(m_malo.mse_resid), np.sqrt(m_bueno.mse_resid)],
    "R2_ajustado": [m_malo.rsquared_adj, m_bueno.rsquared_adj],
}).round(4)

El modelo correcto tiene un error residual **menor** ($4.01$ vs. $5.38$) pese a
gastar un parámetro más: los saltos reales **no** son iguales, y forzarlos a serlo
cuesta ajuste.

## Cómo funcionan las variables dummy

Con $m$ niveles se crean $m-1$ variables indicadoras y un nivel queda como
**referencia** (aquí `Sin`). Cada fila tiene un 1 en la columna de su categoría y 0
en las demás; las viviendas *Sin* piscina tienen 0 en ambas —son el punto de partida.

In [ ]:
# La matriz de diseño que statsmodels construye a partir de la fórmula
diseno = pd.DataFrame(m_bueno.model.exog, columns=m_bueno.model.exog_names)
diseno.insert(0, "Piscina", viviendas.Piscina.values)
diseno.head(8)

In [ ]:
print(m_bueno.summary())

## Interpretación

$$\widehat{Precio} = 31.22 + 1.178\,Area + 10.94\,\mathbb{1}[Pequena] + 36.30\,\mathbb{1}[Grande]$$

* **`Area` = 1.178:** cada m² adicional agrega 1.18 millones **a igual tipo de
  piscina**.
* **`Pequena` = 10.94:** comparando dos viviendas **de la misma área**, una con
  piscina pequeña vale en promedio **10.94 millones más** que una sin piscina.
  **El coeficiente es una diferencia respecto a la referencia**, no un valor absoluto.
* **`Grande` = 36.30:** misma lectura. Nótese que $36.30 \neq 2\times10.94$: los
  saltos **no** son iguales, exactamente lo que la codificación 1-2-3 habría
  impuesto por la fuerza.
* El intercepto ($31.22$) corresponde a una vivienda de $0$ m² **sin** piscina: la
  referencia queda absorbida en el intercepto.

Geométricamente: **tres rectas paralelas** (misma pendiente en `Area`, distinto
intercepto según la piscina).

In [ ]:
fig, ax = plt.subplots()
colores = {"Sin": "steelblue", "Pequena": "seagreen", "Grande": "firebrick"}
viviendas["ajustado"] = m_bueno.fittedvalues
for nivel, g in viviendas.groupby("Piscina"):
    g = g.sort_values("Area")
    ax.scatter(g.Area, g.Precio, s=55, color=colores[nivel], label=nivel)
    ax.plot(g.Area, g.ajustado, color=colores[nivel], lw=2)
ax.set(xlabel="Área (m²)", ylabel="Precio (millones)",
       title="Variables dummy: tres rectas paralelas")
ax.legend(title="Piscina")
plt.show()

In [ ]:
# Cambiar la referencia NO cambia el modelo: solo cambia CON QUIÉN se compara.
m_ref = smf.ols('Precio ~ Area + C(Piscina, Treatment(reference="Grande"))',
                data=viviendas).fit()
print(m_ref.params.round(4))
print("\nR2 con referencia 'Sin'   :", round(m_bueno.rsquared, 6))
print("R2 con referencia 'Grande':", round(m_ref.rsquared, 6))

In [ ]:
# Para preguntar "¿IMPORTA la piscina?" NO se miran las t individuales de las dummies
# (eso son comparaciones por pares). Se compara el modelo con y sin el factor completo:
m_solo_area = smf.ols("Precio ~ Area", data=viviendas).fit()
sm.stats.anova_lm(m_solo_area, m_bueno)

$F = 85.2$ con 2 y 11 gl, $p < 0.001$: el factor `Piscina` en conjunto aporta.

---
# 20. Interacciones (introducción)

Hasta ahora los efectos eran **aditivos**: el efecto de $X_1$ era el mismo
cualquiera fuese el valor de $X_2$. A veces eso es falso.

$$Y = \beta_0 + \beta_1X_1 + \beta_2X_2 + \beta_3X_1X_2 + \epsilon$$

Reagrupando:

$$Y = \beta_0 + \underbrace{(\beta_1 + \beta_3X_2)}_{\text{efecto de } X_1}X_1 + \beta_2X_2 + \epsilon$$

> **El efecto de una variable depende del valor de la otra.**

En `trees` esto tiene sentido físico: si $V \propto d^2h$, el efecto de un pie más
de altura **es mayor en un árbol grueso** que en uno delgado.

In [ ]:
modelo_inter = smf.ols("Volume ~ Girth * Height", data=trees).fit()   # = G + H + G:H
print(modelo_inter.summary().tables[1])
print("\nR2 ajustado:", round(modelo_inter.rsquared_adj, 4),
      "| sigma:", round(np.sqrt(modelo_inter.mse_resid), 4))

In [ ]:
b = modelo_inter.params
pd.DataFrame({
    "Height": [65, 76, 87],
    "efecto_de_Girth": [b["Girth"] + b["Girth:Height"] * h for h in [65, 76, 87]],
}).round(3)

En un árbol de 65 pies, una pulgada más de diámetro vale $\approx 2.9$ pies³; en
uno de 87 pies, vale $\approx 5.9$ pies³: **más del doble**.

In [ ]:
fig, ax = plt.subplots()
gs = np.linspace(8, 21, 50)
for h, c in zip([65, 76, 87], ["steelblue", "seagreen", "firebrick"]):
    nd = pd.DataFrame({"Girth": gs, "Height": h})
    ax.plot(gs, modelo_inter.predict(nd), color=c, lw=2, label=f"Height = {h}")
ax.set(xlabel="Girth", ylabel="Volume estimado",
       title="Interacción: las pendientes ya no son paralelas")
ax.legend()
plt.show()

**Advertencias para la clase siguiente:**

* Si se incluye `X1:X2`, se deben mantener `X1` y `X2` (principio de jerarquía),
  aunque sus $p$ individuales sean grandes.
* Con interacción, el coeficiente de `Girth` ya **no** es "el efecto de Girth": es el
  efecto cuando `Height = 0`, que aquí no significa nada. **Centrar** las predictoras
  devuelve la interpretabilidad.
* Los términos de interacción tienen VIF alto por construcción; eso es estructural.
* Nótese que el modelo log-log capturaba la misma idea de forma más limpia (el
  logaritmo convierte el producto en suma) y con supuestos mejor cumplidos.

---
# 21. Comparación de modelos

In [ ]:
M0 = smf.ols("Volume ~ 1",              data=trees).fit()
M1 = smf.ols("Volume ~ Girth",          data=trees).fit()
M2 = smf.ols("Volume ~ Height",         data=trees).fit()
M3 = smf.ols("Volume ~ Girth + Height", data=trees).fit()
M4 = smf.ols("np.log(Volume) ~ np.log(Girth) + np.log(Height)", data=trees).fit()

def rmse(pred):
    return float(np.sqrt(np.mean((trees.Volume - pred) ** 2)))

pd.DataFrame({
    "modelo": ["M0: Volume ~ 1", "M1: Volume ~ Girth", "M2: Volume ~ Height",
               "M3: Volume ~ Girth + Height", "M4: log(V) ~ log(G) + log(H)"],
    "k_predictoras": [0, 1, 1, 2, 2],
    "R2":     [m.rsquared     for m in (M0, M1, M2, M3, M4)],
    "R2_aj":  [m.rsquared_adj for m in (M0, M1, M2, M3, M4)],
    "sigma":  [np.sqrt(m.mse_resid) for m in (M0, M1, M2, M3, M4)],
    "AIC":    [m.aic for m in (M0, M1, M2, M3, M4)],
    "RMSE_pies3": [rmse(M0.fittedvalues), rmse(M1.fittedvalues), rmse(M2.fittedvalues),
                   rmse(M3.fittedvalues), rmse(np.exp(M4.fittedvalues))],
}).round(4)

In [ ]:
# Modelos anidados: cada paso añade una variable
sm.stats.anova_lm(M0, M1, M3)

## Cómo leer esta tabla

* **M0 (nulo)** es la línea de base: predecir siempre 30.17 pies³. $\hat\sigma$ es la
  desviación estándar de `Volume`.
* **M1 (solo `Girth`)** explica el 93.5 %: el diámetro solo ya hace casi todo el
  trabajo.
* **M2 (solo `Height`)** explica el 35.8 %: la altura sola es un predictor pobre.
* **M3** añade `Height` a M1 y gana $1.3$ puntos de $R^2$ ($p = 0.0145$). Ganancia
  real pero **modesta**.
* **M4 (log-log)** es el mejor en la escala original (RMSE $2.41$) y el único con
  supuestos cumplidos. Su AIC **no** es comparable con los demás (respuesta distinta).

## ⚠️ Los $R^2$, los AIC y las cifras no deciden solos

**No elija un modelo solo porque tenga el $R^2$ más alto.** Pesan tanto o más:

1. **¿Se cumplen los supuestos?** Un modelo con $R^2 = 0.95$ y residuos curvados es
   peor que uno con $R^2 = 0.93$ y residuos limpios: el primero se equivoca de forma
   **sistemática**, y eso no se arregla con más datos.
2. **¿Tienen sentido los coeficientes?** Un signo imposible es una alarma, aunque el
   ajuste sea excelente.
3. **¿Para qué es el modelo?** Predecir → comparar RMSE (idealmente fuera de muestra,
   con validación cruzada). Entender → interpretabilidad y estabilidad.
4. **¿Es parsimonioso?** Entre dos equivalentes, el más simple gana.
5. **Comparabilidad.** AIC/BIC solo se comparan entre modelos con **la misma variable
   respuesta** y **los mismos datos**.

**Decisión para `trees`:** nos quedamos con **M4**, el modelo log-log. Ajusta mejor,
cumple los supuestos, no tiene puntos influyentes y sus coeficientes recuperan la
geometría del problema.

---
# 22. Sincronización R ↔ Python

| Tarea | R | Python |
|---|---|---|
| Leer datos | `read.csv("trees.csv")` | `pd.read_csv("trees.csv")` |
| Estructura | `str(df)` | `df.info()` |
| Descriptivos | `summary(df)` | `df.describe()` |
| Correlaciones | `cor(df)` | `df.corr()` |
| Matriz de dispersión | `GGally::ggpairs(df)` / `pairs(df)` | `sns.pairplot(df)` |
| **Ajustar el modelo** | `lm(y ~ x1 + x2, data = d)` | `smf.ols("y ~ x1 + x2", data=d).fit()` |
| Resumen | `summary(m)` | `m.summary()` |
| Coeficientes | `coef(m)` | `m.params` |
| Errores estándar | `summary(m)$coefficients[,2]` | `m.bse` |
| Estadísticos $t$ / $p$ | `summary(m)$coefficients[,3:4]` | `m.tvalues` / `m.pvalues` |
| Intervalos de confianza | `confint(m)` | `m.conf_int()` |
| $R^2$ / $R^2$ ajustado | `summary(m)$r.squared` / `$adj.r.squared` | `m.rsquared` / `m.rsquared_adj` |
| $\hat\sigma$ | `summary(m)$sigma` | `np.sqrt(m.mse_resid)` |
| Estadístico $F$ global | `summary(m)$fstatistic` | `m.fvalue`, `m.f_pvalue` |
| Tabla ANOVA | `anova(m)` | `sm.stats.anova_lm(m, typ=1)` |
| Comparar anidados | `anova(m1, m2)` | `sm.stats.anova_lm(m1, m2)` |
| AIC | `AIC(m)` | `m.aic` |
| Ajustados | `fitted(m)` | `m.fittedvalues` |
| Residuos ordinarios | `residuals(m)` | `m.resid` |
| Residuos estandarizados | `rstandard(m)` | `infl.resid_studentized_internal` |
| Residuos studentizados | `rstudent(m)` | `infl.resid_studentized_external` |
| Leverage | `hatvalues(m)` | `infl.hat_matrix_diag` |
| Distancia de Cook | `cooks.distance(m)` | `infl.cooks_distance[0]` |
| Objeto de influencia | (integrado) | `infl = m.get_influence()` |
| Q-Q plot | `qqnorm()` + `qqline()` | `sm.qqplot(resid, line="45", fit=True)` |
| Shapiro-Wilk | `shapiro.test(r)` | `scipy.stats.shapiro(r)` |
| Breusch-Pagan | `lmtest::bptest(m)` | `het_breuschpagan(m.resid, m.model.exog)` |
| Durbin-Watson | `lmtest::dwtest(m)` | `durbin_watson(m.resid)` (sin p-valor) |
| VIF | `car::vif(m)` | `variance_inflation_factor(X.values, i)` |
| Predicción puntual | `predict(m, newdata)` | `m.predict(newdata)` |
| IC + IP | `predict(m, nd, interval="confidence"/"prediction")` | `m.get_prediction(nd).summary_frame()` |
| Variable categórica | factor automático en `lm()` | `C(var)` o `C(var, Treatment(reference="..."))` |
| Matriz de diseño | `model.matrix(m)` | `m.model.exog` / `m.model.exog_names` |
| Interacción | `y ~ x1 * x2` | `"y ~ x1 * x2"` |

## Diferencias que hay que conocer

1. **Índices.** R indexa desde **1**; Python desde **0**. La observación "número 31"
   de R es el índice **30** en `pandas`. En este cuaderno forzamos
   `trees.index = range(1, 32)` para que ambos materiales hablen el mismo idioma.
2. **Breusch-Pagan.** `lmtest::bptest()` usa por defecto la versión **studentizada
   (Koenker)**, que es exactamente la que devuelve `het_breuschpagan()`. **Coinciden**
   ($BP = 2.4681$, $p = 0.2911$). Con `bptest(m, studentize = FALSE)` obtendría
   $1.7375$: otra prueba.
3. **Durbin-Watson.** `durbin_watson()` devuelve **solo el estadístico**;
   `lmtest::dwtest()` devuelve además el p-valor.
4. **`anova()`.** En R es secuencial (tipo I) por defecto; `anova_lm(m, typ=1)`
   replica ese comportamiento. Con `typ=2` o `typ=3` los números cambian.
5. **`summary()`.** El de `statsmodels` incluye de entrada Durbin-Watson, Jarque-Bera,
   asimetría, curtosis y número de condición, que en R hay que pedir aparte.
6. **La columna índice del CSV.** `trees.csv` trae una primera columna sin nombre con
   los números de fila. En R aparece como `X`; en Python como `Unnamed: 0`.
   **Hay que eliminarla en ambos lados** o entrará como predictora.
7. **Categóricas.** R convierte un `character` a factor automáticamente dentro de
   `lm()`. `statsmodels` también con fórmulas, pero conviene ser explícito con `C()`.
8. **`sklearn` vs `statsmodels`.** `LinearRegression` ajusta el mismo modelo pero **no
   da errores estándar, ni $t$, ni $p$, ni intervalos**: está pensado para predicción,
   no para inferencia. Para esta clase, **`statsmodels`**.

## Control de calidad

La celda siguiente verifica que los resultados de este cuaderno coinciden con los
del material de R. Si algo falla, lo más probable es que la columna índice del CSV
no se haya eliminado.

In [ ]:
esperado = {
    "beta_0":        -57.9877,
    "beta_Girth":      4.7082,
    "beta_Height":     0.3393,
    "R2":              0.9480,
    "R2_ajustado":     0.9442,
    "sigma":           3.8818,
    "F":             254.9723,
    "SCE":           421.9214,
    "shapiro_W":       0.9732,
    "BP_LM":           2.4681,
    "VIF":             1.3692,
    "cook_obs31":      0.6052,
}
obtenido = {
    "beta_0":       modelo.params["Intercept"],
    "beta_Girth":   modelo.params["Girth"],
    "beta_Height":  modelo.params["Height"],
    "R2":           modelo.rsquared,
    "R2_ajustado":  modelo.rsquared_adj,
    "sigma":        np.sqrt(modelo.mse_resid),
    "F":            modelo.fvalue,
    "SCE":          (modelo.resid ** 2).sum(),
    "shapiro_W":    stats.shapiro(r_stud)[0],
    "BP_LM":        het_breuschpagan(modelo.resid, modelo.model.exog)[0],
    "VIF":          variance_inflation_factor(X_vif.values, 1),
    "cook_obs31":   cook[30],
}
control = pd.DataFrame({"esperado": esperado, "obtenido": obtenido})
control["ok"] = np.isclose(control.esperado, control.obtenido, rtol=1e-3)
print(control.round(4))
print("\n¿TODO COINCIDE CON R?", bool(control.ok.all()))

---
# 23. Ejercicios propuestos

**E1 (nivelación).** Ajuste `Volume ~ Height`. Interprete la pendiente. Luego ajuste
`Volume ~ Girth + Height` y compare el coeficiente de `Height` en ambos. ¿Aumentó o
disminuyó? Explique por qué usando el argumento de la sección 3.

**E2 (efecto parcial).** Repita el procedimiento de Frisch-Waugh-Lovell de la sección
3 pero para `Height`. Verifique que obtiene $0.3393$.

**E3 (ANOVA).** Construya la tabla ANOVA a mano para `Volume ~ Girth`. Verifique que
$F = t^2$.

**E4 (predicción).** Un ingeniero mide un árbol de $Girth = 19$ y $Height = 85$. Dé la
predicción con el modelo lineal y con el log-log. ¿Cuál reportaría y con qué intervalo?

**E5 (extrapolación).** Prediga el volumen de un árbol con $Girth = 25$, $Height = 90$.
¿Confía en el resultado? ¿Por qué?

**E6 (diagnóstico).** Ajuste `Volume ~ Girth` y haga el diagnóstico completo. ¿El
problema de curvatura es peor o mejor que con dos predictoras? ¿Qué le dice eso?

**E7 (influencia).** Identifique la observación más influyente del modelo log-log y
haga el análisis de sensibilidad. ¿Cambia alguna conclusión?

**E8 (transformación alternativa).** Ajuste `Volume ~ I(Girth**2 * Height)` —una sola
predictora construida desde la teoría del cono. Compare su RMSE con los de M3 y M4.

**E9 (categóricas).** En `viviendas`, cambie la referencia a `Pequena` y reinterprete
los tres coeficientes. Verifique que los valores ajustados son idénticos.

**E10 (interacción).** Ajuste `Precio ~ Area * Piscina`. ¿Qué significaría que la
interacción fuera significativa? Dibuje las rectas.

**E11 (aplicado).** Tome un conjunto de datos de su interés profesional con una
respuesta continua y al menos dos predictoras. Recorra el ciclo completo: pregunta →
datos → modelo → resultados → diagnóstico → decisión. Entregue un cuaderno
reproducible con una conclusión de máximo un párrafo dirigida a alguien que no sabe
estadística.

---
# 24. Glosario rápido

| Término | Definición operativa |
|---|---|
| **Coeficiente parcial** | Cambio promedio en $Y$ por unidad de $X_j$ **con las demás predictoras del modelo fijas**. |
| **$R^2$** | $1 - SCE/SCT$. Proporción de variabilidad de $Y$ explicada. Nunca baja al agregar predictoras. |
| **$R^2$ ajustado** | $R^2$ penalizado por el número de parámetros. Puede bajar. Para comparar modelos con distinto $k$. |
| **$\hat\sigma$ (RSE)** | Desviación típica de los residuos, en unidades de $Y$. Interpretable. |
| **Leverage $h_{ii}$** | Cuánto pesa $y_i$ en su propio ajustado. Depende **solo** de las $X$. Suma $= p$; promedio $= p/n$. |
| **Residuo studentizado** | Residuo en escala comparable, con $\sigma$ estimado sin esa observación. Sigue una $t_{n-p-1}$. |
| **Distancia de Cook** | Cuánto se mueven **todos** los ajustados si se quita la observación $i$. Combina residuo y leverage. |
| **VIF** | $1/(1-R_j^2)$. Factor de inflación de la **varianza** del coeficiente. $\sqrt{VIF}$ infla el error estándar. |
| **Intervalo de confianza** | Para la **media** de $Y$ en $x_0$. Se angosta con $n$. |
| **Intervalo de predicción** | Para **una observación** en $x_0$. Siempre más ancho; piso $\approx\pm1.96\hat\sigma$. |
| **Homocedasticidad** | Varianza del error constante a lo largo de los ajustados. |
| **Elasticidad** | En modelos log-log, el coeficiente es el % de cambio en $Y$ por 1 % de cambio en $X$. |

---
# 25. Referencias

* Montgomery, D. C., Peck, E. A. & Vining, G. G. (2021). *Introduction to Linear
  Regression Analysis* (6.ª ed.). Wiley.
* Chatterjee, S. & Hadi, A. S. (2012). *Regression Analysis by Example* (5.ª ed.). Wiley.
* James, G., Witten, D., Hastie, T. & Tibshirani, R. (2021). *An Introduction to
  Statistical Learning* (2.ª ed.). Springer.
* Fox, J. & Weisberg, S. (2019). *An R Companion to Applied Regression* (3.ª ed.). Sage.
* Hoaglin, D. C. & Welsch, R. E. (1978). The Hat Matrix in Regression and ANOVA.
  *The American Statistician*, 32(1), 17–22.
* Ryan, T. A., Joiner, B. L. & Ryan, B. F. (1976). *Minitab Student Handbook*. Duxbury.
  [fuente original del conjunto `trees`]
* Seabold, S. & Perktold, J. (2010). statsmodels: Econometric and statistical modeling
  with Python. *Proceedings of the 9th Python in Science Conference*.

In [ ]:
import sys, platform
print("Python     :", sys.version.split()[0], "|", platform.system())
for mod in (np, pd, sm):
    print(f"{mod.__name__:12s}:", mod.__version__)
import scipy, matplotlib
print("scipy       :", scipy.__version__)
print("matplotlib  :", matplotlib.__version__)